# Section 3.2: Missing Value Handling - Your Code Integration

This notebook shows exactly how to replace your hardcoded missing value handling with the production pipeline.

**Your original approach:** 5 separate hardcoded blocks handling different features
**Pipeline approach:** 1 configuration dict + 2 function calls

## BEFORE: Your Original Hardcoded Code

```python
# Block 1: employment_duration
data['employment_duration_missing'] = data['employment_duration'].isna().astype(int)
data['employment_duration'] = data['employment_duration'].fillna(data['employment_duration'].median())

# Block 2: loan_int_rate
data['loan_int_rate_missing'] = data['loan_int_rate'].isna().astype(int)
data['loan_int_rate'] = data['loan_int_rate'].fillna(data['loan_int_rate'].mean())

# Block 3: loan_amnt
data = data.dropna(subset=['loan_amnt'])

# Block 4: historical_default
data['historical_default_missing'] = (data['historical_default'].isna().astype(int))

# Block 5: Current_loan_status
data = data.dropna(subset=['Current_loan_status'])
```

**Problems:**
- ❌ 5 separate hardcoded blocks (not scalable)
- ❌ No audit trail or metrics
- ❌ No analysis before handling
- ❌ Difficult to test or modify
- ❌ Reasoning embedded in comments

## AFTER: Pipeline Version

Replace all that with configuration + 2 function calls

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Any
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

### Step 1: Configuration (Replaces your 5 hardcoded blocks)

In [ ]:
# ==============================================================================
# MISSING VALUE HANDLING CONFIGURATION
# ==============================================================================
# This single config replaces all 5 of your hardcoded blocks

MISSING_VALUE_CONFIG = {
    'features': {
        # Your Block 1: employment_duration (median impute + indicator)
        'employment_duration': {
            'strategy': 'impute',
            'method': 'median',
            'custom_value': None,
            'create_indicator': True,
            'indicator_name': 'employment_duration_missing',
            'threshold_pct': 25.0,
            'description': 'Likely predictive of credit risk, missing ~2.8%; median-impute + indicator'
        },
        
        # Your Block 2: loan_int_rate (mean impute + indicator)
        'loan_int_rate': {
            'strategy': 'impute',
            'method': 'mean',
            'custom_value': None,
            'create_indicator': True,
            'indicator_name': 'loan_int_rate_missing',
            'threshold_pct': 25.0,
            'description': 'Likely predictive of credit risk; missing ~9.56%; mean-impute + indicator'
        },
        
        # Your Block 3: loan_amnt (drop rows)
        'loan_amnt': {
            'strategy': 'drop',
            'method': None,
            'custom_value': None,
            'create_indicator': False,
            'indicator_name': None,
            'threshold_pct': 5.0,
            'description': 'Retain feature; 1 missing value inconsequential; drop rows with missing'
        },
        
        # Your Block 4: historical_default (indicator only, extreme missingness)
        'historical_default': {
            'strategy': 'indicator_only',
            'method': None,
            'custom_value': None,
            'create_indicator': True,
            'indicator_name': 'historical_default_missing',
            'threshold_pct': 100.0,
            'description': 'Extreme missingness ~63.64%; create indicator only, no imputation'
        },
        
        # Your Block 5: Current_loan_status (drop rows)
        'Current_loan_status': {
            'strategy': 'drop',
            'method': None,
            'custom_value': None,
            'create_indicator': False,
            'indicator_name': None,
            'threshold_pct': 5.0,
            'description': 'Retain feature; 4 missing values inconsequential; drop rows with missing'
        }
    },
    'analysis_before': True,
    'fail_if_analysis_issues': False,
}

### Step 2: Define Functions

In [ ]:
def analyze_missingness(df: pd.DataFrame, config: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Analyze missingness patterns BEFORE making any changes.
    This is crucial for understanding your data before handling.
    """
    if config is None:
        config = MISSING_VALUE_CONFIG
    
    analysis = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'features_with_missing': {},
        'features_with_no_missing': [],
        'total_missing_cells': 0,
        'warnings': []
    }
    
    logger.info("\n" + "="*70)
    logger.info("MISSINGNESS ANALYSIS (BEFORE IMPUTATION)")
    logger.info("="*70)
    logger.info(f"\nDataset shape: {len(df)} rows × {len(df.columns)} columns")
    
    # Check each column for missing values
    for col in df.columns:
        n_missing = df[col].isna().sum()
        pct_missing = (n_missing / len(df)) * 100
        
        if n_missing > 0:
            analysis['features_with_missing'][col] = {
                'count': int(n_missing),
                'percent': round(pct_missing, 2),
                'dtype': str(df[col].dtype)
            }
            analysis['total_missing_cells'] += n_missing
        else:
            analysis['features_with_no_missing'].append(col)
    
    logger.info(f"\nColumns with missing values: {len(analysis['features_with_missing'])}")
    logger.info(f"Columns with no missing values: {len(analysis['features_with_no_missing'])}")
    logger.info(f"Total missing cells: {analysis['total_missing_cells']}")
    
    if analysis['features_with_missing']:
        logger.info(f"\nDetailed breakdown:")
        for col, stats in sorted(analysis['features_with_missing'].items(), 
                                   key=lambda x: x[1]['percent'], reverse=True):
            logger.info(f"  {col:30s}: {stats['count']:5d} missing ({stats['percent']:6.2f}%) [{stats['dtype']}]")
    
    logger.info(f"\n" + "="*70)
    return analysis

In [ ]:
def handle_missing_values(
    df: pd.DataFrame, 
    config: Dict[str, Any] = None,
    analysis: Dict[str, Any] = None
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Handle missing values according to feature-specific strategies.
    This function executes your configuration.
    """
    if config is None:
        config = MISSING_VALUE_CONFIG
    
    if analysis is None:
        analysis = analyze_missingness(df, config)
    
    audit = {
        'status': 'STARTED',
        'rows_input': len(df),
        'rows_removed': 0,
        'columns_input': len(df.columns),
        'columns_output': len(df.columns),
        'features_processed': [],
        'indicators_created': [],
        'details': {},
        'errors': [],
        'warnings': []
    }
    
    df_clean = df.copy()
    
    logger.info("\n" + "="*70)
    logger.info("MISSING VALUE HANDLING EXECUTION")
    logger.info("="*70)
    
    # Process each feature
    for feature_name, feature_config in config['features'].items():
        if feature_name not in df_clean.columns:
            logger.warning(f"✗ {feature_name}: Column not found")
            continue
        
        n_missing_before = df_clean[feature_name].isna().sum()
        
        if n_missing_before == 0:
            logger.info(f"✓ {feature_name:30s}: No missing values")
            audit['features_processed'].append(feature_name)
            continue
        
        strategy = feature_config['strategy']
        logger.info(f"\n{'─'*70}")
        logger.info(f"Feature: {feature_name}")
        logger.info(f"  Missing before: {n_missing_before} ({(n_missing_before/len(df_clean)*100):.2f}%)")
        logger.info(f"  Strategy: {strategy}")
        
        detail = {
            'strategy': strategy,
            'missing_before': int(n_missing_before),
            'missing_after': 0,
            'rows_removed': 0,
            'imputation_method': feature_config.get('method'),
            'indicator_created': False
        }
        
        # STRATEGY 1: IMPUTE
        if strategy == 'impute':
            method = feature_config['method']
            
            if method == 'median':
                fill_value = df_clean[feature_name].median()
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Median imputation (value: {fill_value:.2f})")
            
            elif method == 'mean':
                fill_value = df_clean[feature_name].mean()
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Mean imputation (value: {fill_value:.2f})")
            
            elif method == 'mode':
                fill_value = df_clean[feature_name].mode()[0]
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Mode imputation (value: {fill_value})")
        
        # STRATEGY 2: DROP
        elif strategy == 'drop':
            rows_before = len(df_clean)
            df_clean = df_clean.dropna(subset=[feature_name])
            rows_removed = rows_before - len(df_clean)
            detail['rows_removed'] = int(rows_removed)
            audit['rows_removed'] += rows_removed
            logger.info(f"  Action: Dropped {rows_removed} rows with missing values")
            logger.info(f"  Dataset: {rows_before} → {len(df_clean)} rows")
        
        # STRATEGY 3: INDICATOR ONLY
        elif strategy == 'indicator_only':
            logger.info(f"  Action: No imputation; creating missing indicator only")
        
        # CREATE MISSING INDICATOR
        if feature_config['create_indicator']:
            indicator_name = feature_config['indicator_name']
            if strategy == 'indicator_only':
                df_clean[indicator_name] = (df[feature_name].isna()).astype(int)
            else:
                df_clean[indicator_name] = (df[feature_name].isna()).astype(int)
            
            n_indicated = df_clean[indicator_name].sum()
            audit['indicators_created'].append(indicator_name)
            detail['indicator_created'] = True
            detail['indicator_name'] = indicator_name
            detail['indicator_count'] = int(n_indicated)
            
            logger.info(f"  Indicator: Created '{indicator_name}' ({n_indicated} = 1)")
        
        n_missing_after = df_clean[feature_name].isna().sum()
        detail['missing_after'] = int(n_missing_after)
        audit['features_processed'].append(feature_name)
        audit['details'][feature_name] = detail
        
        if n_missing_after == 0:
            logger.info(f"  Result: ✓ No missing values remain")
        else:
            logger.info(f"  Result: ⚠ {n_missing_after} missing values remain")
    
    audit['rows_output'] = len(df_clean)
    audit['columns_output'] = len(df_clean.columns)
    
    # Validation
    logger.info(f"\n" + "─"*70)
    logger.info("VALIDATION")
    remaining_missing = df_clean.isnull().sum()
    unexpected_missing = remaining_missing[remaining_missing > 0]
    
    if len(unexpected_missing) > 0:
        logger.warning(f"⚠ Unexpected missing values found:")
        for col, count in unexpected_missing.items():
            logger.warning(f"  {col}: {count} missing")
    else:
        logger.info(f"✓ No unexpected missing values remain")
    
    audit['status'] = 'SUCCESS'
    
    # Summary
    logger.info(f"\n" + "="*70)
    logger.info("MISSING VALUE HANDLING SUMMARY")
    logger.info("="*70)
    logger.info(f"Status: {audit['status']}")
    logger.info(f"Features processed: {len(audit['features_processed'])}")
    logger.info(f"Indicators created: {len(audit['indicators_created'])}")
    if audit['indicators_created']:
        logger.info(f"  → {', '.join(audit['indicators_created'])}")
    logger.info(f"Rows: {audit['rows_input']} → {audit['rows_output']} (removed: {audit['rows_removed']})")
    logger.info(f"Columns: {audit['columns_input']} → {audit['columns_output']} (added: {len(audit['indicators_created'])})")
    logger.info(f"="*70)
    
    return df_clean, audit

### Step 3: Load Sample Data

In [ ]:
# Create sample data matching your real patterns
np.random.seed(42)
n_rows = 10000

sample_data = pd.DataFrame({
    'customer_id': range(1, n_rows + 1),
    'age': np.random.randint(20, 70, n_rows),
    'income': np.random.randint(20000, 150000, n_rows),
    'employment_duration': np.random.uniform(0, 40, n_rows),
    'loan_int_rate': np.random.uniform(3, 20, n_rows),
    'loan_amnt': np.random.randint(5000, 100000, n_rows),
    'historical_default': np.random.uniform(0, 1, n_rows),
    'Current_loan_status': np.random.choice([0, 1, 2], n_rows),
    'loan_approved': np.random.choice([0, 1], n_rows)
})

# Introduce missing values matching your percentages
missing_indices = np.random.choice(n_rows, int(n_rows * 0.028), replace=False)
sample_data.loc[missing_indices, 'employment_duration'] = np.nan

missing_indices = np.random.choice(n_rows, int(n_rows * 0.0956), replace=False)
sample_data.loc[missing_indices, 'loan_int_rate'] = np.nan

sample_data.loc[0, 'loan_amnt'] = np.nan

missing_indices = np.random.choice(n_rows, int(n_rows * 0.6364), replace=False)
sample_data.loc[missing_indices, 'historical_default'] = np.nan

sample_data.loc[np.random.choice(n_rows, 4, replace=False), 'Current_loan_status'] = np.nan

print("Sample data created with missing values matching your patterns")
print(f"Shape: {sample_data.shape}")

### Step 4: Run Analysis (BEFORE handling)

In [ ]:
# This replaces your manual inspection - gives you baseline metrics
analysis = analyze_missingness(sample_data, MISSING_VALUE_CONFIG)

### Step 5: Run Handling (Execute your configuration)

In [ ]:
# This executes all your 5 hardcoded blocks in one coordinated pipeline
data_cleaned, audit_trail = handle_missing_values(sample_data, MISSING_VALUE_CONFIG, analysis)

### Step 6: Inspect Results

In [ ]:
# Check the audit trail
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print(f"\nStatus: {audit_trail['status']}")
print(f"\nRows:")
print(f"  Input:   {audit_trail['rows_input']:6d}")
print(f"  Output:  {audit_trail['rows_output']:6d}")
print(f"  Removed: {audit_trail['rows_removed']:6d}")

print(f"\nColumns:")
print(f"  Input:  {audit_trail['columns_input']:6d}")
print(f"  Output: {audit_trail['columns_output']:6d}")
print(f"  Added (indicators): {len(audit_trail['indicators_created']):6d}")

print(f"\nIndicators created:")
for indicator in audit_trail['indicators_created']:
    count = data_cleaned[indicator].sum()
    print(f"  - {indicator:30s}: {count:4d} rows marked as originally missing")

print(f"\nData quality check:")
remaining_missing = data_cleaned.isnull().sum().sum()
if remaining_missing == 0:
    print(f"  ✓ PASSED: No missing values remain")
else:
    print(f"  ✗ FAILED: {remaining_missing} missing cells remain")

print("\n" + "="*70)

### Step 7: Drop customer_id (Now safe after duplicate checking)

In [ ]:
# Now that we're done with customer_id for duplicate checks,
# we can drop it (it provides little value for modeling)
if 'customer_id' in data_cleaned.columns:
    data_cleaned = data_cleaned.drop(columns=['customer_id'])
    print("✓ customer_id column dropped")

print(f"\nFinal shape: {data_cleaned.shape}")
print(f"\nFinal columns:")
print(data_cleaned.columns.tolist())

## Side-by-Side Comparison

### BEFORE (Your original code)
```python
# employment_duration has ~2.8% missing values.
# Feature is likely predictive of credit risk, so we will retain.
# Missing values will be median-imputed with a missingness indicator.
data['employment_duration_missing'] = data['employment_duration'].isna().astype(int)
data['employment_duration'] = data['employment_duration'].fillna(data['employment_duration'].median())

# loan_int_rate has ~9.56% missing values
# Feature is likely predictive of credit risk, certainly a collinear feature.  Retain.
# Missing values will be mean-imputed with a missingness indicator.
data['loan_int_rate_missing'] = data['loan_int_rate'].isna().astype(int)
data['loan_int_rate'] = data['loan_int_rate'].fillna(data['loan_int_rate'].mean())

# loan_amnt has 1 missing values
# Will retain feature.  One missing value of our sampling size is inconsequential.
# Missing value will be dropped.
data = data.dropna(subset=['loan_amnt'])

# historical_default has ~63.64% missing values
# Missing indicator
data['historical_default_missing'] = (data['historical_default'].isna().astype(int))

# Current_loan_status has four missing values
# Will retain feature.  Four missing values of our sampling size is inconsequential.
# Missing value will be dropped.
data = data.dropna(subset=['Current_loan_status'])
```

**Problems:**
- 5 separate blocks (hard to maintain)
- No audit trail
- No analysis before handling
- Reasoning in comments
- Can't easily modify or test

---

### AFTER (Pipeline version)

**Cell 1: Configuration**
```python
MISSING_VALUE_CONFIG = {
    'features': {
        'employment_duration': {'strategy': 'impute', 'method': 'median', 'create_indicator': True, ...},
        'loan_int_rate': {'strategy': 'impute', 'method': 'mean', 'create_indicator': True, ...},
        'loan_amnt': {'strategy': 'drop', ...},
        'historical_default': {'strategy': 'indicator_only', 'create_indicator': True, ...},
        'Current_loan_status': {'strategy': 'drop', ...}
    }
}
```

**Cell 2: Functions** (copy from above or import from module)

**Cell 3: Execute**
```python
analysis = analyze_missingness(data, MISSING_VALUE_CONFIG)
data_cleaned, audit = handle_missing_values(data, MISSING_VALUE_CONFIG, analysis)
```

**Benefits:**
- ✅ Single configuration (easy to understand)
- ✅ Comprehensive audit trail
- ✅ Analysis before handling
- ✅ Reasoning in configuration (self-documenting)
- ✅ Easy to modify and test
- ✅ Scales to any number of features

---

## What Changed?

| Aspect | Before | After |
|--------|--------|-------|
| **Code blocks** | 5 separate hardcoded blocks | 1 config dict |
| **Lines of code** | ~15 | ~200 (but reusable) |
| **Audit trail** | None | Comprehensive |
| **Analysis** | Manual | Automated |
| **Maintainability** | Low | High |
| **Testing** | Difficult | Easy |
| **Scalability** | Poor | Excellent |

---

## Next Steps

1. ✅ **Duplicates handled** (with audit trail)
2. ✅ **Missing values handled** (with audit trail and indicators)
3. ➜ **Next: Outlier handling** (similar pipeline structure)
4. Then: Data type corrections
5. Then: Value corrections
6. Then: Feature engineering
7. Then: EDA on cleaned data
8. Then: Model development